# 1.使用FUNCTION_CALL模式

In [2]:
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.tools.structured import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType, create_tool_calling_agent, AgentExecutor
from langchain_community.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

#读取配置文件里的信息
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

#获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)

search_tool = Tool(
    func=search.run,
    name="Search",
    description="用于检索互联网上的信息",
)

#获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

#提供提示词模板（以ChatPromptTEmplate为例）
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个乐于助人的ai助手，根据用户的提问，必要时调用Search工具，使用互联网检索数据"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

#获取Agent的实例，create_tool_calling_agent()
agent = create_tool_calling_agent(
    llm=chat_model,
    prompt=prompt_template,
    tools=[search_tool],
)

#获取AgentExecutor的实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
)

result = agent_executor.invoke({"input": "今天的湘潭天气怎么样？"})

print(result)



> Entering new AgentExecutor chain...

Invoking: `Search` with `湘潭天气预报`


[{'title': '湘潭天气预报,湘潭7天天气预报,湘潭15天天气预报,湘潭天气查询', 'url': 'https://www.weather.com.cn/weather/101250201.shtml', 'content': '台风路径\n空间天气\n图片\n专题\n环境\n旅游\n碳中和\n气象科普\n一带一路\n产创平台\n\n热门城市\n\n热门景点\n\n选择省市\n\n<<返回\n全国\n\n周边城市\n\n周边景点\n\n本地乡镇\n\n热门城市\n\n选择洲际\n\n# 17日（今天）\n\n小雨\n\n10℃\n\n<3级\n\n# 18日（明天）\n\n小雨\n\n13℃/10℃\n\n<3级\n\n# 19日（后天）\n\n小雨\n\n14℃/10℃\n\n<3级\n\n# 20日（周五）\n\n小雨转多云\n\n14℃/12℃\n\n<3级\n\n# 21日（周六）\n\n小雨\n\n17℃/13℃\n\n<3级\n\n# 22日（周日）\n\n小雨转中雨\n\n17℃/15℃\n\n<3级\n\n# 23日（周一）\n\n大雨转小雨\n\n17℃/13℃\n\n<3级\n\n### 蓝天预报综合天气现象、能见度、空气质量等因子，预测未来一周的天空状况。\n\n天凉，湿度大，较易感冒。\n\n有降水，推荐您在室内进行休闲运动。\n\n外出需远离过敏源，适当采取防护措施。\n\n建议着厚外套加毛衣等服装。\n\n有雨，雨水和泥水会弄脏爱车。\n\n辐射弱，涂擦SPF8-12防晒护肤品。\n\n天凉，湿度大，较易感冒。\n\n有降水，推荐您在室内进行休闲运动。\n\n外出需远离过敏源，适当采取防护措施。\n\n建议着厚外套加毛衣等服装。\n\n有雨，雨水和泥水会弄脏爱车。\n\n辐射弱，涂擦SPF8-12防晒护肤品。\n\n天凉，湿度大，较易感冒。\n\n有降水，推荐您在室内进行休闲运动。\n\n外出需远离过敏源，适当采取防护措施。\n\n建议着厚外套加毛衣等服装。\n\n有雨，雨水和泥水会弄脏爱车。\n\n辐射弱，涂擦SPF8-12防晒护肤品。\n\n天凉，湿度大，较易感冒。\n\n有

# 注意：agent_scratchpad必须声明，它用于存储和传递Agent的思考过程。比如，在调用链式工具时（如先搜索天气再推荐行程）， agent_scratchpad 保留所有历史步骤，避免上下文丢失。format方法会将intermediate_steps转换为特定格式的字符串，并赋值给agent_scratchpad变量。如果不传递intermediate_steps参数，会导致KeyError: 'intermediate_steps'错误。

# 2.使用ReAct模式

## 1.使用PromptTemplate


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.tools.structured import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType, create_tool_calling_agent, AgentExecutor, \
    create_react_agent
from langchain_community.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

#读取配置文件里的信息
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

#获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)

search_tool = Tool(
    func=search.run,
    name="Search",
    description="用于检索互联网上的信息",
)

#获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

#提供提示词模板（以ChatPromptTEmplate为例）
template = """Answer the following questions as best you can. You have access to the following tools:
        {tools}
        Use the following format:
        Question: the input question you must answer
        Thought: you should always think about what to do
        Action: the action to take, should be one of [{tool_names}]
        Action Input: the input to the action
        Observation: the result of the action
        ... (this Thought/Action/Action Input/Observation can repeat N times)
        Thought: I now know the final answer
        Final Answer: the final answer to the original input question

        Begin!

        Question: {input}
        Thought:{agent_scratchpad}"""
prompt_template = PromptTemplate.from_template(
    template=template,
)

#获取Agent的实例，create_react_agent()
agent = create_react_agent(
    llm=chat_model,
    prompt=prompt_template,
    tools=[search_tool],
)

#获取AgentExecutor的实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
)

result = agent_executor.invoke({"input": "今天的湘潭天气怎么样？"})

print(result)

方式2

In [ ]:
from langchain_classic import hub
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.tools.structured import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType, create_tool_calling_agent, AgentExecutor, \
    create_react_agent
from langchain_community.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

#读取配置文件里的信息
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

#获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)

search_tool = Tool(
    func=search.run,
    name="Search",
    description="用于检索互联网上的信息",
)

#获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

#提供提示词模板（以ChatPromptTEmplate为例）
prompt_template = hub.pull("hwchase17/react")
#获取Agent的实例，create_react_agent()
agent = create_react_agent(
    llm=chat_model,
    prompt=prompt_template,
    tools=[search_tool],
)

#获取AgentExecutor的实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
)

result = agent_executor.invoke({"input": "今天的湘潭天气怎么样？"})

print(result)

## 2.使用ChatPromptTemplate

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.tools.structured import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType, create_tool_calling_agent, AgentExecutor, \
    create_react_agent
from langchain_community.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

#读取配置文件里的信息
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

#获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)

search_tool = Tool(
    func=search.run,
    name="Search",
    description="用于检索互联网上的信息",
)

#获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

#提供提示词模板（以ChatPromptTEmplate为例）
template = """Answer the following questions as best you can. You have access to the following tools:
        {tools}
        Use the following format:
        Question: the input question you must answer
        Thought: you should always think about what to do
        Action: the action to take, should be one of [{tool_names}]
        Action Input: the input to the action
        Observation: the result of the action
        ... (this Thought/Action/Action Input/Observation can repeat N times)
        Thought: I now know the final answer
        Final Answer: the final answer to the original input question

        Begin!"""
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个乐于助人的ai助手，根据用户的提问，必要时调用Search工具，使用互联网检索数据"),
    ("system",template),
    ("human", "{input}"),
    ("system", "{agent_scratchpad}"),
])

#获取Agent的实例，create_react_agent()
agent = create_react_agent(
    llm=chat_model,
    prompt=prompt_template,
    tools=[search_tool],
)

#获取AgentExecutor的实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
    handle_parsing_errors=True
)

result = agent_executor.invoke({"input": "今天的湘潭天气怎么样？"})

print(result)



> Entering new AgentExecutor chain...
Thought: 我需要查找今天湘潭的天气情况。  
Action: Search  
Action Input: "湘潭天气预报"  [{'title': '湘潭天气预报,湘潭7天天气预报,湘潭15天天气预报,湘潭天气查询', 'url': 'https://www.weather.com.cn/weather/101250201.shtml', 'content': '台风路径\n空间天气\n图片\n专题\n环境\n旅游\n碳中和\n气象科普\n一带一路\n产创平台\n\n热门城市\n\n热门景点\n\n选择省市\n\n<<返回\n全国\n\n周边城市\n\n周边景点\n\n本地乡镇\n\n热门城市\n\n选择洲际\n\n# 17日（今天）\n\n小雨\n\n10℃\n\n<3级\n\n# 18日（明天）\n\n小雨\n\n13℃/10℃\n\n<3级\n\n# 19日（后天）\n\n小雨\n\n14℃/10℃\n\n<3级\n\n# 20日（周五）\n\n小雨转多云\n\n14℃/12℃\n\n<3级\n\n# 21日（周六）\n\n小雨\n\n17℃/13℃\n\n<3级\n\n# 22日（周日）\n\n小雨转中雨\n\n17℃/15℃\n\n<3级\n\n# 23日（周一）\n\n大雨转小雨\n\n17℃/13℃\n\n<3级\n\n### 蓝天预报综合天气现象、能见度、空气质量等因子，预测未来一周的天空状况。\n\n天凉，湿度大，较易感冒。\n\n有降水，推荐您在室内进行休闲运动。\n\n外出需远离过敏源，适当采取防护措施。\n\n建议着厚外套加毛衣等服装。\n\n有雨，雨水和泥水会弄脏爱车。\n\n辐射弱，涂擦SPF8-12防晒护肤品。\n\n天凉，湿度大，较易感冒。\n\n有降水，推荐您在室内进行休闲运动。\n\n外出需远离过敏源，适当采取防护措施。\n\n建议着厚外套加毛衣等服装。\n\n有雨，雨水和泥水会弄脏爱车。\n\n辐射弱，涂擦SPF8-12防晒护肤品。\n\n天凉，湿度大，较易感冒。\n\n有降水，推荐您在室内进行休闲运动。\n\n外出需远离过敏源，适当采取防护措施。\n\n建议着厚外套加毛衣等服装。\n\n有雨，雨水和泥水会弄脏爱车。\n\n辐射弱，涂擦SP

### 小结：
1.传统方式，相较于通用方式来讲，不用提供提示词模板。

2.对于通用方式来讲，

FUNCTION_CALL模式：在创建Agent时，推荐大家使用ChatPromptTemplate

ReAct模式：在创建Agent时，大家可以使用ChatPromptTemplate、PromptTemplate。但相较来讲，推荐大家使用PromptTemplate
